In [17]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
nltk.download('punkt_tab')


import spacy


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/julienrm/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [18]:
df_fr = pd.read_csv('data/small_vocab_fr.txt', sep='\t', names=['text'])
df_fr.head()

,text
0,new jersey est parfois calme pendant l' automn...
1,les états-unis est généralement froid en juill...
2,"california est généralement calme en mars , et..."
3,"les états-unis est parfois légère en juin , et..."
4,"votre moins aimé fruit est le raisin , mais mo..."


In [19]:
df_en = pd.read_csv('data/small_vocab_en.txt', sep='\t', names=['text'])
df_en.head()

,text
0,"new jersey is sometimes quiet during autumn , ..."
1,the united states is usually chilly during jul...
2,"california is usually quiet during march , and..."
3,the united states is sometimes mild during jun...
4,"your least liked fruit is the grape , but my l..."


In [20]:
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def preprocess_text(text, remove_stopwords=False, language='french', remove_punctuation=False):
    """
    Preprocess text by cleaning and tokenizing
    """
    # Basic text cleaning
    text = text.strip() # .lower()
    def fix_punctuation_spacing(text):
        # Apostrophe: no spaces around it
        if text.find("'") != -1:
            text = text.replace(" '", "'").replace("' ", "'")

        # Hyphen in compound words: no spaces around it
        # Em dash (—) or en dash (–): space before and after for sentence breaks
        if text.find("-") != -1:
            # First handle spaced dashes (likely sentence breaks)
            text = text.replace(" - ", " — ")  # Convert to em dash
            text = text.replace(" -", " —").replace("- ", "— ")
            
            # Replace em dashes back to spaced format
            text = text.replace("—", " — ")
            
            # Clean up multiple spaces around em dashes
            text = re.sub(r'\s*—\s*', ' — ', text)
        
        # Comma: no space before, one space after
        if text.find(",") != -1:
            text = text.replace(" ,", ",")
            # Add space after comma if not already there
            text = re.sub(r',(?!\s)', ', ', text)
            # Fix multiple spaces after comma
            text = text.replace(",  ", ", ")

        # Period: no space before, one space after (except end of text)
        if text.find(".") != -1:
            text = text.replace(" .", ".")
            # Add space after period if not already there and not at end
            text = re.sub(r'\.(?!\s|$)', '. ', text)
            # Fix multiple spaces after period
            text = text.replace(".  ", ". ")
        
        # Semicolon: no space before, one space after
        if text.find(";") != -1:
            text = text.replace(" ;", ";")
            text = re.sub(r';(?!\s)', '; ', text)
            text = text.replace(";  ", "; ")
        
        # Colon: no space before, one space after
        if text.find(":") != -1:
            text = text.replace(" :", ":")
            text = re.sub(r':(?!\s)', ': ', text)
            text = text.replace(":  ", ": ")
        
        # Question mark: no space before, one space after
        if text.find("?") != -1:
            text = text.replace(" ?", "?")
            text = re.sub(r'\?(?!\s|$)', '? ', text)
            text = text.replace("?  ", "? ")
        
        # Exclamation mark: no space before, one space after
        if text.find("!") != -1:
            text = text.replace(" !", "!")
            text = re.sub(r'!(?!\s|$)', '! ', text)
            text = text.replace("!  ", "! ")
        
        # Opening parenthesis: one space before (if not at start), no space after
        if text.find("(") != -1:
            text = re.sub(r'(?<!\s)(?<!^)\(', ' (', text)  # Add space before if not already there
            text = text.replace("( ", "(")  # Remove space after
            text = text.replace("  (", " (")  # Fix double spaces
        
        # Closing parenthesis: no space before, one space after (if not at end)
        if text.find(")") != -1:
            text = text.replace(" )", ")")
            text = re.sub(r'\)(?!\s|$|[.,;:!?])', ') ', text)  # Add space after unless at end or before punctuation
            text = text.replace(")  ", ") ")
        
        # Clean up any multiple spaces
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()

    text = fix_punctuation_spacing(text)

    # Remove digits
    # cleaned_text = ''.join(char for char in text if not char.isdigit())
    
    # Remove punctuation
    # if remove_punctuation:
    #     for punctuation in string.punctuation:
    #         cleaned_text = cleaned_text.replace(punctuation, ' ')

    # Tokenize
    word_tokens = word_tokenize(text, language=language)
    
    # if asked to remove stopwords
    if remove_stopwords:
        print("Removing stopwords")
        # Remove stop words
        if language == 'french':
            stop_words = set(stopwords.words('french'))
        else:
            stop_words = set(stopwords.words('english'))
    
        tokens_cleaned = [w for w in word_tokens if w not in stop_words and len(w) > 0]
    
        return tokens_cleaned
    
    # # Load relevant language model
    # if language == 'french':
        
    #     nlp = spacy.load('fr_core_news_sm')
    # else:
    #     nlp = spacy.load('en_core_web_sm')

    # def process_text(text):
    #     # this is processing part.
    #     doc = nlp(text)

    #     # Filtering step
    #     filtered_tokens = [token.text for token in doc if not token.is_stop]

    #     print("Filtered Tokens:", filtered_tokens)
    word_tokens = [wt for wt in word_tokens if len(wt) > 0]
    # print(word_tokens)

    return word_tokens

# Apply preprocessing to French data
df_fr['tokens'] = df_fr['text'].apply(lambda x: preprocess_text(x, remove_stopwords= False, language='french'))

# Instantiating the TfidfVectorizer
tf_idf_vectorizer_fr = TfidfVectorizer()
# Training it on the texts
weighted_df_fr = pd.DataFrame(tf_idf_vectorizer_fr.fit_transform(df_fr['text']).toarray(),
                    columns = tf_idf_vectorizer_fr.get_feature_names_out())

# Apply preprocessing to English data
df_en['tokens'] = df_en['text'].apply(lambda x: preprocess_text(x, remove_stopwords= False, language='english'))

# Instantiating the TfidfVectorizer
tf_idf_vectorizer_en = TfidfVectorizer()

# Training it on the texts
weighted_df_en = pd.DataFrame(tf_idf_vectorizer_en.fit_transform(df_en['text']).toarray(),
                    columns = tf_idf_vectorizer_en.get_feature_names_out())

print("French preprocessing complete")
print("English preprocessing complete")

# Rename columns to be specific to each language
df_fr_renamed = df_fr.rename(columns={'text': 'text_fr', 'tokens': 'tokens_fr'})
df_en_renamed = df_en.rename(columns={'text': 'text_en', 'tokens': 'tokens_en'})

# Combine side by side
df_fr_en = pd.concat([df_fr_renamed, df_en_renamed], axis=1)

import csv
with open('data/cleaned_texts.csv', 'w', newline='') as csvfile:
    df_fr_en.to_csv(csvfile, index=False)

French preprocessing complete
English preprocessing complete


In [21]:
print("\nFrench:", weighted_df_fr.shape[1], "\nEnglish:", weighted_df_en.shape[1])


French: 321 
English: 196


In [22]:
# Combine side by side

df_fr_en

,text_fr,tokens_fr,text_en,tokens_en
0,new jersey est parfois calme pendant l' automn...,"[new, jersey, est, parfois, calme, pendant, l'...","new jersey is sometimes quiet during autumn , ...","[new, jersey, is, sometimes, quiet, during, au..."
1,les états-unis est généralement froid en juill...,"[les, états-unis, est, généralement, froid, en...",the united states is usually chilly during jul...,"[the, united, states, is, usually, chilly, dur..."
2,"california est généralement calme en mars , et...","[california, est, généralement, calme, en, mar...","california is usually quiet during march , and...","[california, is, usually, quiet, during, march..."
3,"les états-unis est parfois légère en juin , et...","[les, états-unis, est, parfois, légère, en, ju...",the united states is sometimes mild during jun...,"[the, united, states, is, sometimes, mild, dur..."
4,"votre moins aimé fruit est le raisin , mais mo...","[votre, moins, aimé, fruit, est, le, raisin, ,...","your least liked fruit is the grape , but my l...","[your, least, liked, fruit, is, the, grape, ,,..."
...,...,...,...,...
137855,"la france est jamais occupée en mars , et il e...","[la, france, est, jamais, occupée, en, mars, ,...","france is never busy during march , and it is ...","[france, is, never, busy, during, march, ,, an..."
137856,"l' inde est parfois belle au printemps , et il...","[l'inde, est, parfois, belle, au, printemps, ,...","india is sometimes beautiful during spring , a...","[india, is, sometimes, beautiful, during, spri..."
137857,"l' inde est jamais mouillé pendant l' été , ma...","[l'inde, est, jamais, mouillé, pendant, l'été,...","india is never wet during summer , but it is s...","[india, is, never, wet, during, summer, ,, but..."
137858,"la france est jamais froid en janvier , mais i...","[la, france, est, jamais, froid, en, janvier, ...","france is never chilly during january , but it...","[france, is, never, chilly, during, january, ,..."


In [23]:
df_fr_en.text_fr[137855]

'la france est jamais occupée en mars , et il est parfois agréable en septembre .'

In [24]:
df_fr_en.tokens_fr[137855]

['la',
 'france',
 'est',
 'jamais',
 'occupée',
 'en',
 'mars',
 ',',
 'et',
 'il',
 'est',
 'parfois',
 'agréable',
 'en',
 'septembre',
 '.']

In [25]:
weighted_df_en

,am,and,animal,animals,apple,apples,april,are,aren,august,...,when,where,white,why,winter,wonderful,would,yellow,you,your
0,0.0,0.180260,0.0,0.0,0.000000,0.0,0.366932,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
1,0.0,0.162972,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
2,0.0,0.175169,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
3,0.0,0.175852,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
4,0.0,0.000000,0.0,0.0,0.304045,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.255303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137855,0.0,0.185612,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
137856,0.0,0.191898,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
137857,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.377138,0.0,0.0,0.0,0.0,0.000000
137858,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000


In [26]:
df_fr_en['text_fr'][23]

"paris est doux pendant l' été , mais il est généralement occupé en avril ."

In [27]:
df_fr_en['text_fr']
df_fr_en['text_en']

0         new jersey is sometimes quiet during autumn , ...
1         the united states is usually chilly during jul...
2         california is usually quiet during march , and...
3         the united states is sometimes mild during jun...
4         your least liked fruit is the grape , but my l...
                                ...                        
137855    france is never busy during march , and it is ...
137856    india is sometimes beautiful during spring , a...
137857    india is never wet during summer , but it is s...
137858    france is never chilly during january , but it...
137859    the orange is her favorite fruit , but the ban...
Name: text_en, Length: 137860, dtype: object

In [28]:
df_fr_en["tokens_fr"]

0         [new, jersey, est, parfois, calme, pendant, l'...
1         [les, états-unis, est, généralement, froid, en...
2         [california, est, généralement, calme, en, mar...
3         [les, états-unis, est, parfois, légère, en, ju...
4         [votre, moins, aimé, fruit, est, le, raisin, ,...
                                ...                        
137855    [la, france, est, jamais, occupée, en, mars, ,...
137856    [l'inde, est, parfois, belle, au, printemps, ,...
137857    [l'inde, est, jamais, mouillé, pendant, l'été,...
137858    [la, france, est, jamais, froid, en, janvier, ...
137859    [l'orange, est, son, fruit, préféré, ,, mais, ...
Name: tokens_fr, Length: 137860, dtype: object

In [29]:
from LSTM_translator import train_translator_from_tokens, load_translator_for_inference, test_translation

In [30]:
# # Your data structure: df with columns ['tokens_fr', 'tokens_en']
# # Example: df.iloc[0]['tokens_fr'] = ['new', 'jersey', 'est', 'parfois', 'calme', ...]

# # Train the model
# translator, history = train_translator_from_tokens(df_fr_en, validation_size=1000)

# # Save for later use
# translator.save_model('lstm_translator')

# # Test some translations
# test_translation(translator, df_fr_en, n_examples=5)


In [31]:
translator = load_translator_for_inference('lstm_translator')

# # Method 1: Preprocess then translate tokens manually
# french_tokens = translator.preprocess_french_phrase("Bonjour, comment allez-vous ?")
# english_tokens = translator.translate_tokens(french_tokens)
# print(english_tokens)

# Method 2: Translate entire sentence directly
english_tokens = translator.translate_sentence("c'est meilleur la banane ?")
print(english_tokens)

Loading model from lstm_translator...


ValueError: Unknown layer: 'NotEqual'. Please ensure you are using a `keras.utils.custom_object_scope` and that this object is included in the scope. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.

In [ ]:

translator = load_translator_for_inference('lstm_translator')

# Translate new French tokens but change to words for vocabulary
french_tokens = ['bonjour', 'comment', 'allez', 'vous']
french_tokens = ["C'est meilleur la banane ?"]
english_tokens = translator.translate_tokens(french_tokens)
print(english_tokens)  # ['hello', 'how', 'are', 'you']

Loading model from lstm_translator...


ValueError: Unknown layer: 'NotEqual'. Please ensure you are using a `keras.utils.custom_object_scope` and that this object is included in the scope. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.

In [ ]:
  # Method 2: Translate entire sentence directly
  english_tokens = translator.translate_sentence("Bonjour, comment allez-vous ?")